In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-feedback-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 단건 테스트 — generate_daily_feedback()

실제 서비스 함수를 그대로 호출합니다.  
CSV 트랜잭션 데이터 → 분석 → RAG → 잔소리 피드백 전 과정이 실행됩니다.

In [ ]:
from catcher_llm.services.consumption_feedback.daily_feedback import generate_daily_feedback

result = generate_daily_feedback(
    member_id=1,
    analysis_date="2024-01-15",
    previous_date="2024-01-14"
)

if result.error:
    print("오류:", result.error)
else:
    print("[피드백]")
    print(result.feedback.scolding_message)
    print()
    print("[내일 미션]", result.feedback.tomorrow_mission)
    print()
    print("[오늘 총지출]", result.daily_analysis.stable_metrics.today_total if result.daily_analysis else "-")

# 2. target 함수 정의

In [ ]:
def target(inputs: dict):
    """
    LangSmith evaluate()에 넘길 target 함수.
    inputs: {"member_id": int, "analysis_date": str, "previous_date": str}
    """
    result = generate_daily_feedback(
        member_id=inputs["member_id"],
        analysis_date=inputs["analysis_date"],
        previous_date=inputs["previous_date"]
    )

    if result.error:
        return {"answer": f"[ERROR] {result.error}", "today_total": 0, "top_category": ""}

    # 피드백 평가에 필요한 소비 수치도 함께 넘긴다 (groundedness 검증용)
    today_total = (
        result.daily_analysis.stable_metrics.today_total
        if result.daily_analysis else 0
    )
    top_category = (
        result.daily_analysis.stable_metrics.worst_category
        if result.daily_analysis else ""
    )

    return {
        "answer": result.feedback.scolding_message,
        "tomorrow_mission": result.feedback.tomorrow_mission,
        "today_total": today_total,
        "top_category": str(top_category)
    }

# 3. LangSmith Dataset 생성

서로 다른 유저 × 날짜 조합 5개.  
`ground_truth`는 "이 피드백이 갖춰야 할 특성"을 기술합니다.

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-feedback-daily-eval"

examples = [
    {
        "inputs": {"member_id": 1, "analysis_date": "2024-01-15", "previous_date": "2024-01-14"},
        "outputs": {"expected_trait": "배달 관련 지출을 언급하고, 내일 실천 가능한 행동을 1개 이상 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 2, "analysis_date": "2024-01-20", "previous_date": "2024-01-19"},
        "outputs": {"expected_trait": "간편결제 또는 온라인 소비를 언급하고, 구체적인 절약 행동을 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 3, "analysis_date": "2024-02-05", "previous_date": "2024-02-04"},
        "outputs": {"expected_trait": "소비가 적은 날이라도 지출 데이터에 근거한 피드백을 제공해야 한다."}
    },
    {
        "inputs": {"member_id": 4, "analysis_date": "2024-02-10", "previous_date": "2024-02-09"},
        "outputs": {"expected_trait": "구독 관련 지출을 언급하고, 중복 구독 점검을 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 1, "analysis_date": "2024-03-01", "previous_date": "2024-02-29"},
        "outputs": {"expected_trait": "월초 소비 패턴을 파악하고, 이번 달 절약 시작점을 제안해야 한다."}
    },
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Catcher LLM 일간 피드백 품질 평가"
    )
    for ex in examples:
        client.create_example(
            inputs=ex["inputs"],
            outputs=ex["outputs"],
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(examples)}개)")

# 4. Evaluator 정의

| Evaluator | 방식 | 기준 |
|---|---|---|
| `groundedness` | LLM judge | 피드백이 실제 소비 수치를 언급하는가 |
| `actionability` | LLM judge | 내일 당장 실행 가능한 행동을 제안하는가 |
| `personalization` | LLM judge | 이 유저의 특성(페르소나/목표)에 맞는가 |
| `contains_amount` | Heuristic | 구체적 금액이 포함되어 있는가 |

In [ ]:
import re
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def groundedness_evaluator(run, example):
    """피드백 내 수치·카테고리가 실제 소비 데이터에 근거하는가 (0~1)"""
    answer = run.outputs.get("answer", "")
    today_total = run.outputs.get("today_total", 0)
    top_category = run.outputs.get("top_category", "")

    prompt = f"""
아래 피드백이 실제 소비 데이터에 근거하는지 평가해줘.
피드백이 오늘 총지출({today_total:,}원), 최다 지출 카테고리({top_category})를
구체적으로 언급하거나 반영하면 높은 점수야.
소비 데이터와 무관한 일반론만 있으면 낮은 점수야.
0~1 숫자 하나만 출력해.

피드백: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "groundedness", "score": float(score)}
    except ValueError:
        return {"key": "groundedness", "score": 0.0}


def actionability_evaluator(run, example):
    """내일 당장 실행 가능한 행동 제안이 있는가 (0~1)"""
    answer = run.outputs.get("answer", "")
    tomorrow = run.outputs.get("tomorrow_mission", "")

    prompt = f"""
아래 피드백과 내일 미션을 보고, 사용자가 내일 당장 실천할 수 있는
구체적인 행동 제안이 포함되어 있는지 0~1로 평가해줘.
"절약하세요" 같은 추상적 조언은 낮게, 카테고리/방법이 명확하면 높게.
숫자 하나만 출력해.

피드백: {answer}
내일 미션: {tomorrow}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "actionability", "score": float(score)}
    except ValueError:
        return {"key": "actionability", "score": 0.0}


def personalization_evaluator(run, example):
    """expected_trait(기대 특성)을 얼마나 충족하는가 (0~1)"""
    answer = run.outputs.get("answer", "")
    expected_trait = example.outputs.get("expected_trait", "")

    prompt = f"""
아래 피드백이 다음 기대 특성을 얼마나 충족하는지 0~1로 평가해줘.

기대 특성: {expected_trait}
피드백: {answer}

숫자 하나만 출력해.
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "personalization", "score": float(score)}
    except ValueError:
        return {"key": "personalization", "score": 0.0}


def contains_amount_evaluator(run, example):
    """구체적 금액(숫자+원)이 피드백에 포함되어 있는가 (0 or 1)"""
    answer = run.outputs.get("answer", "")
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}


print("evaluator 4개 정의 완료")

# 5. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[
        groundedness_evaluator,
        actionability_evaluator,
        personalization_evaluator,
        contains_amount_evaluator,
    ],
    experiment_prefix="daily-feedback-v1"
)